# 🎛️ 04 · Model Training & Selection

> **Chapter 4.** Optuna TPE search over XGBoost hyperparameters,
> PR-AUC-only objective, median pruning. The final model is what the
> tuning history converged to, not a lucky single trial.

## Why Optuna TPE instead of random search

The pre-audit pipeline used 14 random-search trials with a composite
objective `0.65·ROC + 0.35·PR`. Two problems:

1. **Random search wastes trials** on bad hyperparameter regions.
2. **A composite objective is arbitrary** — what if we had weighted
   0.5 / 0.5? 0.8 / 0.2? No principled answer.

The rewrite uses `optuna.samplers.TPESampler(multivariate=True)` —
models the joint distribution over hyperparameters, samples
promising regions. Combined with `MedianPruner` (kills clearly bad
trials early), we get **40 trials in the same wall-clock budget as
the old 14**. Objective is pure PR-AUC: under ~24 % class imbalance
PR-AUC is informative, ROC-AUC is not.

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
history = pd.read_csv(REPO / "results/metrics/xgboost_tuning_history.csv")
best = pd.read_csv(REPO / "results/metrics/xgboost_best_params.csv").iloc[0]
print(f"Optuna trials total: {len(history)}")
print(f"Best val PR-AUC: {history['val_pr_auc'].max():.4f}")
print(f"Best val ROC-AUC: {history['val_roc_auc'].max():.4f}")

## Winning hyperparameters

In [ ]:
print("Selected configuration:")
for key in ["max_depth", "learning_rate", "min_child_weight", "subsample",
            "colsample_bytree", "colsample_bylevel", "gamma",
            "reg_alpha", "reg_lambda", "max_delta_step",
            "scale_pos_weight", "best_threshold"]:
    if key in best:
        val = best[key]
        try:
            val_f = float(val)
            print(f"  {key:<22s}  {val_f:.4g}")
        except (TypeError, ValueError):
            print(f"  {key:<22s}  {val}")

## Tuning history — how quickly did the search converge?

In [ ]:
history_sorted = history.sort_values("trial_number") if "trial_number" in history.columns else history
running_best = history_sorted["val_pr_auc"].cummax()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(running_best.values, color="#2c3e50", lw=2, label="running best val PR-AUC")
ax.scatter(range(len(history_sorted)), history_sorted["val_pr_auc"].values,
           s=18, alpha=0.5, color="#e74c3c", label="individual trial")
ax.axhline(history["val_pr_auc"].max(), color="gray", ls="--", alpha=0.6,
           label=f"final best = {history['val_pr_auc'].max():.4f}")
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation PR-AUC")
ax.legend(loc="lower right", frameon=False)
ax.set_title("Optuna TPE search — convergence trace",
             fontweight="bold", loc="left")
fig.tight_layout()
plt.show()

**Key insight.** The running-best curve flattens by trial ~10 and
never gains more than 0.001 after. That's the strongest possible
empirical signal that the model is at a feature-limited ceiling —
no amount of additional hyperparameter compute will move the number
without new information. Phase 2 (ESM-2 + AlphaFold2) is explicitly
about adding new information.

## Threshold sweep

After training, we sweep decision thresholds over [0.2, 0.8] and
pick the one maximizing F1 on *validation* (never on test). The
chosen threshold is then frozen for test scoring.

In [ ]:
curve = pd.read_csv(REPO / "results/metrics/xgboost_val_threshold_curve.csv")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(curve["threshold"], curve["f1"], "o-", ms=3, color="#2c3e50")
best_t = float(best.get("best_threshold", curve.loc[curve["f1"].idxmax(), "threshold"]))
ax.axvline(best_t, color="#e74c3c", ls="--", label=f"chosen threshold = {best_t:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("F1 (validation)")
ax.legend(loc="lower left", frameon=False)
ax.set_title("Validation F1 vs decision threshold",
             fontweight="bold", loc="left")
fig.tight_layout()
plt.show()

## Split metrics at the chosen threshold

In [ ]:
split = pd.read_csv(REPO / "results/metrics/xgboost_split_metrics.csv")
print(split[["split", "threshold", "roc_auc", "pr_auc", "f1", "precision", "recall",
             "brier_loss", "support"]].to_string(index=False))

## Reproduce

```bash
python -m src.training --trials 40 --seed 42
```

The trial budget is intentionally `40`, not `100`. Above 40 the
search cost dominates with no measurable gain — we verified this
experimentally. Seed=42 makes the sampler deterministic so reruns
produce identical histories.